In [1]:
import pandas as pd
import numpy as np
import re
import html
from pathlib import Path

In [2]:
adzuna_raw = pd.read_csv("/Users/manrajsingh/Downloads/adzuna_raw_jobs.csv")
jooble_raw = pd.read_csv("/Users/manrajsingh/Downloads/jooble_raw_jobs.csv")

print("Adzuna Shape: ", adzuna_raw.shape)
print("Jooble shape: ", jooble_raw.shape)

Adzuna Shape:  (4948, 26)
Jooble shape:  (9797, 14)


In [3]:
print("Adzuna Head: ", adzuna_raw.head())
print("Jooble Head: ", jooble_raw.head())

Adzuna Head:                                           description  salary_is_predicted  \
0  Here's your clean, copy-ready job posting: JOB...                    0   
1  About Citylitics Inc. Citylitics delivers pred...                    0   
2   About the company Through the latest techniqu...                    0   
3  Vretta is looking for a Data Analyst to join o...                    0   
4  Data Analyst Role : Full-time, on-site 4 days/...                    0   

  contract_time          id   latitude  \
0     part_time  5703538141  43.648370   
1     full_time  5870640464  43.706102   
2     full_time  5888936887  43.706102   
3     full_time  5721992379  43.706102   
4     full_time  5718161437  43.706102   

                                               adref         title  \
0  eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTcwMzUzODE0MSIsI...  Data Analyst   
1  eyJhbGciOiJIUzI1NiJ9.eyJzIjoiTm9Dem1sX0w4Ukdqd...  Data Analyst   
2  eyJhbGciOiJIUzI1NiJ9.eyJzIjoiTm9Dem1sX0w4Ukdqd...  Data A

In [4]:
def inspect_dataset (df, dataset_names):
    print(f"\n ----{dataset_names}----")
    print("Rows: ", len(df))
    print("Columns: ", len(df.columns))
    print("Exact duplicate rows: ", df.duplicated().sum())

    missing = (df.isna().sum().to_frame("missing_count"))
    missing["missing_percentage"] = (missing["missing_count"]/len(df) * 100).round(2)
    display(missing.sort_values("missing_percentage", ascending=False))

inspect_dataset(adzuna_raw, "Adzuna Raw Dataset")
inspect_dataset(jooble_raw, "Jooble Raw Dataset")


 ----Adzuna Raw Dataset----
Rows:  4948
Columns:  26
Exact duplicate rows:  0


,missing_count,missing_percentage
request_datetime,4498,90.91
contract_type,3979,80.42
salary_max,3448,69.68
salary_min,3433,69.38
contract_time,2574,52.02
latitude,222,4.49
longitude,222,4.49
location.area,0,0.00
company.display_name,0,0.00
company.__CLASS__,0,0.00



 ----Jooble Raw Dataset----
Rows:  9797
Columns:  14
Exact duplicate rows:  0


,missing_count,missing_percentage
salary,5533,56.48
type,4265,43.53
company,235,2.40
title,0,0.00
location,0,0.00
snippet,0,0.00
source,0,0.00
link,0,0.00
updated,0,0.00
id,0,0.00


In [5]:
def clean_text(value):
    if pd.isna(value):
        return np.nan
    
    value = html.unescape(str(value))
    value = re.sub(r"<[^>]+>" , " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    if value.lower() in {"", "nan", "none", "null", "n/a"}:
        return np.nan
    
    return value

def parse_jooble_salary(value):
    result = {"salary_min_annual": np.nan,
              "salary_max_annual": np.nan,
              "salary_period": "Unkown"}
    
    if pd.isna(value):
        return result
    
    text = str(value).lower().strip()
# Normalize thousands separators: 50,000 -> 50000
    text = re.sub(r"(?<=\d),(?=\d{3}\b)", "", text)

    # Determine the pay period from explicit wording.
    if re.search(r"\b(per hour|hourly|/hour|/hr)\b", text):
        multiplier = 37.5 * 52
        period = "Hourly"

    elif re.search(r"\b(per month|monthly|/month)\b", text):
        multiplier = 12
        period = "Monthly"

    elif re.search(r"\b(per week|weekly|/week)\b", text):
        multiplier = 52
        period = "Weekly"

    elif re.search(r"\b(per day|daily|/day)\b", text):
        multiplier = 260
        period = "Daily"

    elif re.search(
        r"\b(annually|annual|yearly|per year|per annum|a year)\b",
        text
    ):
        multiplier = 1
        period = "Annual"

    else:
        # Do not guess the pay period.
        return result

    # Match complete numbers with optional k/m suffixes.
    pattern = r"(?<!\w)(\d+(?:,\d{3})*(?:\.\d+)?)\s*([km]?)(?!\w)"
    matches = re.findall(pattern, text)

    amounts = []

    for number, suffix in matches:
        amount = float(number.replace(",", ""))

        if suffix == "k":
            amount *= 1_000
        elif suffix == "m":
            amount *= 1_000_000

        amounts.append(amount)

    if not amounts:
        return result

    low = amounts[0]
    high = amounts[1] if len(amounts) > 1 else amounts[0]

    if low > high:
        low, high = high, low

    result["salary_min_annual"] = low * multiplier
    result["salary_max_annual"] = high * multiplier
    result["salary_period"] = period

    return result

In [6]:
### Standardizing both datasets


# --------------ADZUNA-----------------
adzuna = pd.DataFrame({"source": "Adzuna",
                       "source_id": adzuna_raw["id"].astype("string"),
                       "title": adzuna_raw["title"].map(clean_text),
                       "company": adzuna_raw["company.display_name"].map(clean_text),
                       "city_raw": adzuna_raw["location.display_name"].map(clean_text),
                       "search_position": adzuna_raw["search_position"].map(clean_text),
                       "description": adzuna_raw["description"].map(clean_text),
                       "salary_min_raw": pd.to_numeric(adzuna_raw["salary_min"], errors="coerce"),
                       "salary_max_raw": pd.to_numeric(adzuna_raw["salary_max"], errors="coerce"),
                       "salary_period": "Annual (assumed by source)",
                       "job_type": adzuna_raw["contract_time"].map(clean_text),
                       "posted_date": pd.to_datetime(adzuna_raw["created"], errors="coerce", utc=True),
                       "url": adzuna_raw["redirect_url"].map(clean_text),
                       "collected_at": pd.to_datetime(adzuna_raw["request_datetime"], errors="coerce", utc=True)})

adzuna["salary_min_annual"] = adzuna["salary_min_raw"].where(adzuna["salary_min_raw"].between(15000, 500000))

adzuna["salary_max_annual"] = adzuna["salary_max_raw"].where(adzuna["salary_max_raw"].between(15000, 500000))

both_endpoints = (adzuna["salary_min_annual"].notna()& adzuna["salary_max_annual"].notna())

reversed_salary = (both_endpoints& (adzuna["salary_min_annual"]> adzuna["salary_max_annual"]))

old_min = adzuna.loc[reversed_salary, "salary_min_annual"].copy()

adzuna.loc[reversed_salary, "salary_min_annual"] = adzuna.loc[reversed_salary, "salary_max_annual"].to_numpy()

adzuna.loc[reversed_salary, "salary_max_annual"] = old_min.to_numpy()

# --------------JOBBLE-----------------

jooble = pd.DataFrame({
    "source": "Jooble",
    "source_id": jooble_raw["id"].astype("string"),
    "title": jooble_raw["title"].map(clean_text),
    "company": jooble_raw["company"].map(clean_text),
    "city_raw": jooble_raw["location"].map(clean_text),
    "search_position": jooble_raw["search_position"].map(clean_text),
    "description": jooble_raw["snippet"].map(clean_text),
    "salary_min_raw": np.nan,
    "salary_max_raw": np.nan,
    "salary_period": "Unknown",
    "job_type": jooble_raw["type"].map(clean_text),
    "posted_date": pd.to_datetime(jooble_raw["updated"], errors="coerce", utc=True),
    "url": jooble_raw["link"].map(clean_text),
    "collected_at": pd.to_datetime(jooble_raw["request_datetime"], errors="coerce", utc=True)})


jooble_salary = jooble_raw["salary"].apply(parse_jooble_salary)

jooble["salary_min_annual"] = jooble_salary.apply(lambda x: x["salary_min_annual"])
jooble["salary_max_annual"] = jooble_salary.apply(lambda x: x["salary_max_annual"])
jooble["salary_period"] = jooble_salary.apply(lambda x: x["salary_period"])

# Retain the original salary text for auditing.
jooble["salary_original"] = jooble_raw["salary"].map(clean_text)

# Retain original salary values for Adzuna too.
adzuna["salary_original"] = (adzuna_raw["salary_min"].astype("string")
    + " - "
    + adzuna_raw["salary_max"].astype("string"))

print("Adzuna standardized:", adzuna.shape)
print("Jooble standardized:", jooble.shape)

Adzuna standardized: (4948, 17)
Jooble standardized: (9797, 17)


In [7]:
mask = (adzuna["salary_min_annual"].notna()& adzuna["salary_max_annual"].notna()& (adzuna["salary_min_annual"]>adzuna["salary_max_annual"]))

old_min = adzuna.loc[mask, "salary_min_annual"].copy()

adzuna.loc[mask, "salary_min_annual"] = (adzuna.loc[mask, "salary_max_annual"].values)

adzuna.loc[mask, "salary_max_annual"] = old_min.values

In [8]:
jobs = pd.concat([adzuna, jooble], ignore_index=True)

def standardize_city(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip().lower()

    if "toronto" in value: 
        return "Toronto"
    elif "vancouver" in value:
        return "Vancouver"
    elif "montreal" in value or "montréal" in value:
        return "Montréal"
    
    return np.nan

def standardize_role(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()

    if "analyst" in value:
        return "Data Analyst"
    elif "scientist" in value:
        return "Data Scientist"
    elif "engineer" in value:
        return "Data Engineer"
    
    return np.nan

jobs["city"] = jobs["city_raw"].apply(standardize_city)
jobs["job_role"] = jobs["search_position"].apply(standardize_role)

# Standardizing common employment type labels
jobs["job_type"] = (jobs["job_type"].str.lower().str.replace("_", " ", regex=False).str.strip())

# Calculate midpoint only when both salary endpoints are available
jobs["salary_mid_annual"] = (jobs["salary_min_annual"] + jobs["salary_max_annual"])/2

# Flag potentially invalid or unusual salary figures for review
jobs["salary_needs_review"] = (jobs["salary_min_annual"].notna()& ((jobs["salary_min_annual"] < 15000)| (jobs["salary_min_annual"] > 500000))) | (jobs["salary_max_annual"].notna()&((jobs["salary_max_annual"] < 15000)| (jobs["salary_max_annual"] > 500000)))

# Keeping only the three intended cities and three intended roles
jobs = jobs[jobs["city"].isin(["Toronto", "Vancouver", "Montréal"])
            & jobs["job_role"].isin(["Data Analyst", "Data Scientist", "Data Engineer"])].copy()

print("Combined records after city and role validation:", len(jobs))

display(jobs[["source", "title", "company", "city", "job_role", "salary_min_annual", "salary_max_annual", "salary_mid_annual"]].head(10))


Combined records after city and role validation: 14444


,source,title,company,city,job_role,salary_min_annual,salary_max_annual,salary_mid_annual
0,Adzuna,Data Analyst,ADF Medical Services Inc.,Toronto,Data Analyst,NaN,NaN,NaN
1,Adzuna,Data Analyst,Citylitics,Toronto,Data Analyst,65000.0,80000.0,72500.0
2,Adzuna,Data Analyst,WOMBO Studios,Toronto,Data Analyst,70000.0,90000.0,80000.0
3,Adzuna,Data Analyst,Vretta,Toronto,Data Analyst,NaN,NaN,NaN
4,Adzuna,Data Analyst,Acquird.io,Toronto,Data Analyst,70000.0,80000.0,75000.0
5,Adzuna,Data Analyst,Konrad,Toronto,Data Analyst,75000.0,100000.0,87500.0
6,Adzuna,Data Analyst,Insight Global,Toronto,Data Analyst,NaN,NaN,NaN
7,Adzuna,Data Analyst,CI Financial,Toronto,Data Analyst,55000.0,95000.0,75000.0
8,Adzuna,Data Analyst,BBDCA Omnicom Advertising Group,Toronto,Data Analyst,NaN,NaN,NaN
9,Adzuna,Data Analyst,CI Financial,Toronto,Data Analyst,55000.0,95000.0,75000.0


In [9]:
for role in ["Data Analyst",
            "Data Scientist",
            "Data Engineer"]:
    print(f"\n--- {role} ---")

    display(jobs.loc[jobs["job_role"] == role,["title", "search_position", "city"]].sample(min(15, (jobs["job_role"] == role).sum()),random_state=42))


--- Data Analyst ---


,title,search_position,city
6551,Senior Customer Success Manager,Data Analyst,Vancouver
7745,Analyste budgétaire - immobilier / Budget Anal...,Data Analyst,Montréal
6267,Online Data Analyst Bengali (CA),Data Analyst,Vancouver
7821,Senior Forward Deployed Engineer (Gemini Enter...,Data Analyst,Montréal
6548,Technical Support Lead,Data Analyst,Vancouver
201,Senior Data Engineer,Data Analyst,Toronto
7442,Business Analyst - Investment Systems,Data Analyst,Montréal
8055,Analyste de crédit,Data Analyst,Montréal
7388,Analyste commercial/analyste commerciale en in...,Data Analyst,Montréal
5793,Technical (Construction) Risk Analyst,Data Analyst,Toronto



--- Data Scientist ---


,title,search_position,city
11215,Business Operations Manager,Data Scientist,Montréal
9653,CNC Press Brake Operator,Data Scientist,Vancouver
9246,Customer Success Manager,Data Scientist,Toronto
9463,Senior Engineer (Hybrid),Data Scientist,Vancouver
9724,Finance Operations Lead,Data Scientist,Vancouver
9834,Head of Technical Recruiting,Data Scientist,Vancouver
754,"Quantitative Research Lead, Construction & Pla...",Data Scientist,Toronto
11245,Accounting Clerk,Data Scientist,Montréal
9099,Advanced Data Scientist Role at Home Depot,Data Scientist,Toronto
753,Junior Analyst,Data Scientist,Toronto



--- Data Engineer ---


,title,search_position,city
13496,Staff Product Manager,Data Engineer,Vancouver
4671,"Assistant Vice-President, Data & Ai",Data Engineer,Montréal
13236,"Software Development Engineer, Web Frameworks",Data Engineer,Vancouver
3356,"Senior Manager, GTM Data Science",Data Engineer,Toronto
13482,Full Stack Developer,Data Engineer,Vancouver
14564,AWS/AI Solutions Architect REQ_130,Data Engineer,Montréal
12279,Principal Engineer in Software Development,Data Engineer,Toronto
14024,Front-End Engineer,Data Engineer,Montréal
13389,Financial Analyst,Data Engineer,Vancouver
13281,"Software Engineer, Agentic Harness for Games (...",Data Engineer,Vancouver


In [10]:
# Remove duplicate IDs within the same source.
jobs = jobs.drop_duplicates(subset=["source", "source_id"],keep="first").copy()

# Remove duplicate URLs within the same source, ignoring missing URLs.
duplicate_url = (jobs["url"].notna()& jobs.duplicated(subset=["source", "url"], keep="first"))

jobs = jobs.loc[~duplicate_url].copy()

# Create normalized keys for identifying potential cross-source duplicates.
def normalize_key(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


jobs["title_key"] = jobs["title"].apply(normalize_key)
jobs["company_key"] = jobs["company"].apply(normalize_key)

# Only compare records with a known title and company.
valid_key = (jobs["title_key"].ne("")& jobs["company_key"].ne(""))

jobs["duplicate_candidate"] = False

duplicate_keys = (jobs.loc[valid_key].groupby(["title_key", "company_key", "city"])["source"].nunique())

cross_source_keys = duplicate_keys[duplicate_keys > 1].index

jobs.loc[valid_key, "duplicate_candidate"] = (jobs.loc[valid_key, ["title_key", "company_key", "city"]].apply(tuple, axis=1).isin(cross_source_keys))

print("Records after within-source deduplication:", len(jobs))
print("Potential cross-source duplicate records:",jobs["duplicate_candidate"].sum())

# Review possible duplicates rather than deleting them automatically.
duplicate_review = jobs.loc[jobs["duplicate_candidate"],["source", "title", "company", "city", "salary_original", "url"]].sort_values(["city", "company", "title"])

display(duplicate_review.head(20))

Records after within-source deduplication: 9780
Potential cross-source duplicate records: 1438


,source,title,company,city,salary_original,url
2333,Adzuna,"Lead Developer, Tools",2K,Montréal,<NA>,https://www.adzuna.ca/details/5841510653?utm_m...
10691,Jooble,"Lead Developer, Tools",2K,Montréal,NaN,https://ca.jooble.org/jdp/-610975952120806657
2341,Adzuna,Lead Programmer | Programmeur(-euse) principal(e),2K,Montréal,<NA>,https://www.adzuna.ca/details/5900460122?utm_m...
10703,Jooble,Lead Programmer | Programmeur(-euse) principal(e),2K,Montréal,NaN,https://ca.jooble.org/jdp/2413153825223519212
2336,Adzuna,"Software Development Manager, Tools & Products",2K,Montréal,<NA>,https://www.adzuna.ca/details/5822421615?utm_m...
10704,Jooble,"Software Development Manager, Tools & Products",2K,Montréal,NaN,https://ca.jooble.org/jdp/6908200536787538887
4710,Adzuna,"Forward Deployed AI Engineer REQ_55, REQ_56",3Pillar,Montréal,<NA>,https://www.adzuna.ca/details/5933526536?utm_m...
14065,Jooble,"Forward Deployed AI Engineer REQ_55, REQ_56",3Pillar,Montréal,NaN,https://ca.jooble.org/jdp/4972337558106979280
4937,Adzuna,Consultant Conversion - Lead Cybersecurity Eng...,860 Morgan Stanley Svcs Canada Co,Montréal,<NA>,https://www.adzuna.ca/details/5910299667?utm_m...
14707,Jooble,Consultant Conversion - Lead Cybersecurity Eng...,860 Morgan Stanley Svcs Canada Co,Montréal,NaN,https://ca.jooble.org/jdp/-7312361711254360252


In [11]:
# Overall Cleaning Summary
cleaning_summary = pd.DataFrame({"Metric": ["Original Adzuna records",
                                            "original Jooble records",
                                            "Total raw records",
                                            "Records retained after cleaning",
                                            "Records with a missing salary midpoint",
                                            "Records flagged for salary review",
                                            "Potential cross-source duplicate records"],
                                "Value": [len(adzuna_raw),
                                          len(jooble_raw),
                                          len(adzuna_raw) + len(jooble_raw),
                                          len(jobs),
                                          jobs["salary_mid_annual"].isna().sum(),
                                          jobs["salary_needs_review"].sum(),
                                          jobs["duplicate_candidate"].sum()]})

display(cleaning_summary)

#Data quality by source, city and job role
quality_report = (jobs.groupby(["source", "city", "job_role"], dropna=False)
                  .agg(total_jobs = ("title", "size"),
                       missing_company = ("company", lambda s: s.isna().sum()),
                       missing_salary = ("salary_mid_annual", lambda s: s.isna().sum()),
                       salary_review_requested = ("salary_needs_review", "sum")).reset_index())

display(quality_report)

,Metric,Value
0,Original Adzuna records,4948
1,original Jooble records,9797
2,Total raw records,14745
3,Records retained after cleaning,9780
4,Records with a missing salary midpoint,6221
5,Records flagged for salary review,20
6,Potential cross-source duplicate records,1438


,source,city,job_role,total_jobs,missing_company,missing_salary,salary_review_requested
0,Adzuna,Montréal,Data Analyst,104,0,93,0
1,Adzuna,Montréal,Data Engineer,188,0,163,0
2,Adzuna,Montréal,Data Scientist,219,0,198,0
3,Adzuna,Toronto,Data Analyst,294,0,194,0
4,Adzuna,Toronto,Data Engineer,878,0,640,0
5,Adzuna,Toronto,Data Scientist,1082,0,796,0
6,Adzuna,Vancouver,Data Analyst,72,0,42,0
7,Adzuna,Vancouver,Data Engineer,248,0,146,0
8,Adzuna,Vancouver,Data Scientist,364,0,204,0
9,Jooble,Montréal,Data Analyst,1050,60,835,0


In [12]:
output_folder = Path("output")
output_folder.mkdir(exist_ok=True)

# Removing helper columns used only for duplicate matching
cleaned_jobs = jobs.drop(columns=["title_key", "company_key"],errors="ignore").copy()

# Saving the cleaned dataset and quality reports
cleaned_jobs.to_csv(output_folder / "cleaned_standardized_jobs.csv",index=False)

cleaning_summary.to_csv(output_folder / "job_data_cleaning_summary.csv",index=False)

quality_report.to_csv(output_folder / "job_data_quality_by_source_city_role.csv",index=False)

duplicate_review.to_csv(output_folder / "cross_source_duplicate_candidates.csv",index=False)

print("Cleaning completed!")
print("Files saved in:", output_folder.resolve())
print("Final dataset shape:", cleaned_jobs.shape)

display(cleaned_jobs.head())

Cleaning completed!
Files saved in: /Users/manrajsingh/Desktop/output
Final dataset shape: (9780, 22)


,source,source_id,title,company,city_raw,search_position,description,salary_min_raw,salary_max_raw,salary_period,...,url,collected_at,salary_min_annual,salary_max_annual,salary_original,city,job_role,salary_mid_annual,salary_needs_review,duplicate_candidate
0,Adzuna,5703538141,Data Analyst,ADF Medical Services Inc.,"Toronto Dominion Centre, City of Toronto",Data Analyst,"Here's your clean, copy-ready job posting: JOB...",NaN,NaN,Annual (assumed by source),...,https://www.adzuna.ca/details/5703538141?utm_m...,2026-10-19 01:51:31.278852+00:00,NaN,NaN,<NA>,Toronto,Data Analyst,NaN,False,False
1,Adzuna,5870640464,Data Analyst,Citylitics,"Toronto, Ontario",Data Analyst,About Citylitics Inc. Citylitics delivers pred...,65000.0,80000.0,Annual (assumed by source),...,https://www.adzuna.ca/details/5870640464?utm_m...,2026-10-19 01:51:31.278852+00:00,65000.0,80000.0,65000.0 - 80000.0,Toronto,Data Analyst,72500.0,False,False
2,Adzuna,5888936887,Data Analyst,WOMBO Studios,"Toronto, Ontario",Data Analyst,About the company Through the latest technique...,70000.0,90000.0,Annual (assumed by source),...,https://www.adzuna.ca/details/5888936887?utm_m...,2026-10-19 01:51:31.278852+00:00,70000.0,90000.0,70000.0 - 90000.0,Toronto,Data Analyst,80000.0,False,False
3,Adzuna,5721992379,Data Analyst,Vretta,"Toronto, Ontario",Data Analyst,Vretta is looking for a Data Analyst to join o...,NaN,NaN,Annual (assumed by source),...,https://www.adzuna.ca/details/5721992379?utm_m...,2026-10-19 01:51:31.278852+00:00,NaN,NaN,<NA>,Toronto,Data Analyst,NaN,False,False
4,Adzuna,5718161437,Data Analyst,Acquird.io,"Toronto, Ontario",Data Analyst,"Data Analyst Role : Full-time, on-site 4 days/...",70000.0,80000.0,Annual (assumed by source),...,https://www.adzuna.ca/details/5718161437?utm_m...,2026-10-19 01:51:31.278852+00:00,70000.0,80000.0,70000.0 - 80000.0,Toronto,Data Analyst,75000.0,False,True


In [13]:
def inspect_dataset(df, dataset_name):
    print(f"\n--- {dataset_name} ---")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Exact duplicate rows:", df.duplicated().sum())

    missing = (df.isna().sum().to_frame("missing_count"))

    missing["missing_percentage"] = (missing["missing_count"] / len(df) * 100).round(2)

    display(missing.sort_values("missing_percentage", ascending=False))

inspect_dataset(adzuna_raw, "Adzuna Raw Data")
inspect_dataset(jooble_raw, "Jooble Raw Data")


--- Adzuna Raw Data ---
Rows: 4948
Columns: 26
Exact duplicate rows: 0


,missing_count,missing_percentage
request_datetime,4498,90.91
contract_type,3979,80.42
salary_max,3448,69.68
salary_min,3433,69.38
contract_time,2574,52.02
latitude,222,4.49
longitude,222,4.49
location.area,0,0.00
company.display_name,0,0.00
company.__CLASS__,0,0.00



--- Jooble Raw Data ---
Rows: 9797
Columns: 14
Exact duplicate rows: 0


,missing_count,missing_percentage
salary,5533,56.48
type,4265,43.53
company,235,2.40
title,0,0.00
location,0,0.00
snippet,0,0.00
source,0,0.00
link,0,0.00
updated,0,0.00
id,0,0.00


In [14]:
print("CLEANED DATASET")
print("Rows:", len(cleaned_jobs))
print("Columns:", len(cleaned_jobs.columns))

print("\nMissing values in important columns:")
important_columns = [
    "source",
    "source_id",
    "title",
    "company",
    "city",
    "job_role",
    "salary_min_annual",
    "salary_max_annual",
    "salary_mid_annual"
]

display(
    cleaned_jobs[important_columns]
    .isna()
    .sum()
    .to_frame("Missing Count")
)

print("\nRows by source:")
display(cleaned_jobs["source"].value_counts(dropna=False))

print("\nRows by city:")
display(cleaned_jobs["city"].value_counts(dropna=False))

print("\nRows by job role:")
display(cleaned_jobs["job_role"].value_counts(dropna=False))

print("\nSalary summary:")
display(
    cleaned_jobs[
        ["salary_min_annual", "salary_max_annual", "salary_mid_annual"]
    ].describe().round(2)
)


CLEANED DATASET
Rows: 9780
Columns: 22

Missing values in important columns:


,Missing Count
source,0
source_id,0
title,0
company,183
city,0
job_role,0
salary_min_annual,6219
salary_max_annual,6219
salary_mid_annual,6221



Rows by source:


source
Jooble    6331
Adzuna    3449
Name: count, dtype: int64


Rows by city:


city
Toronto      4368
Montréal     2928
Vancouver    2484
Name: count, dtype: int64


Rows by job role:


job_role
Data Analyst      3723
Data Scientist    3125
Data Engineer     2932
Name: count, dtype: int64


Salary summary:


,salary_min_annual,salary_max_annual,salary_mid_annual
count,3561.00,3561.00,3559.00
mean,111757.44,144734.26,128241.30
std,48501.23,64473.84,55023.46
min,2000.00,5200.00,4000.00
25%,76000.00,98000.00,87750.00
50%,104000.00,136448.00,120000.00
75%,140000.00,180400.00,160000.00
max,418920.00,425000.00,418920.00
